# Voy a bajar datos de la wikipedia de datos historicos


In [26]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import unicodedata
import numpy as np
import unicodedata


url = "https://es.wikipedia.org/wiki/Anexo:Finalistas_de_la_Euroliga"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9",
}

response = requests.get(url, headers=headers)

if response.ok:
    print("OK → status 200-299")
else:
    print("Error →", response.status_code)


OK → status 200-299


In [27]:
     # Vamos a parsearlo 

soup = BeautifulSoup(response.text, "lxml")    # Toma el HTML crudo (response.text):
                                                    # Usa el parser lxml
                                                         # lxml es:
                                                         # más rápido
                                                         # más estricto
                                                         # más estable
                                                         # repara HTML roto mejor que html.parser
                                                    # Convierte el HTML en un árbol DOM navegable
                                                    # El HTML pasa de ser texto plano a ser un árbol de objetos:
                                                          # Código
                                                          # html
                                                          #  └── body
                                                          #       └── h1
                                                          #            └── "Hola"
                                                    # Crea el objeto soup
                                                         # soup es un árbol completo del HTML, donde puedes hacer:
                                                         # soup.find("table")
                                                         # soup.find_all("a")
                                                         # soup.select(".price")
                                                         # soup.get_text()

In [28]:
print(bool(soup))  # Para saber si no esta vacio


True


In [29]:
# Comprobar si hay tables 
tables = soup.find_all("table")                 # busca TODAS las etiquetas <table> del HTML
print(f"Total tablas; {len(tables)}")           # Esto NO significa:
                                                    # que haya 10 DataFrames
                                                    # que pandas pueda leerlas
                                                    # que todas sean útiles
                                                    # Solo significa: 
                                                    # 👉 El HTML contiene 13 elementos <table>



Total tablas; 3


In [30]:
def tiene_cabeceras(tabla):
    """Debe tener <th> para ser tabla real."""
    return tabla.find("th") is not None

def tiene_filas(tabla):
    """Debe tener varias filas <tr>."""
    filas = tabla.find_all("tr")
    return len(filas) >= 2

def tiene_columnas(tabla):
    """Debe tener varias columnas en la primera fila."""
    primera = tabla.find("tr")
    if not primera:
        return False
    celdas = primera.find_all(["td", "th"])
    return len(celdas) >= 2

def no_es_navbox(tabla):
    """Excluir tablas de navegación, infobox, sidebars, etc."""
    clase = tabla.get("class", [])
    basura = ["navbox", "infobox", "sidebar", "metadata", "vertical-navbox"]
    return not any(b in clase for b in basura)

def tiene_texto_util(tabla):
    """Debe tener texto significativo."""
    texto = tabla.get_text(" ", strip=True)
    return len(texto.split()) > 5

In [31]:
tablas = soup.find_all("table")
print(f"Tablas encontradas: {len(tablas)}")

tablas_validas = []
for t in tablas:
    if (
        tiene_cabeceras(t)
        and tiene_filas(t)
        and tiene_columnas(t)
        and no_es_navbox(t)
        and tiene_texto_util(t)
    ):
        tablas_validas.append(t)

print(f"Tablas válidas detectadas: {len(tablas_validas)}")


Tablas encontradas: 3
Tablas válidas detectadas: 2



Saco las tablas validas

In [32]:
dfs = []

for t in tablas_validas:
    rows = []
    for tr in t.find_all("tr"):
        cells = tr.find_all(["th", "td"])
        cells = [c.get_text(" ", strip=True) for c in cells]
        if cells:
            rows.append(cells)

    # Saltar tablas sin contenido real
    if len(rows) < 2:
        continue

    # Normalizar columnas (rellenar filas cortas)
    max_cols = max(len(r) for r in rows)
    rows = [r + [""] * (max_cols - len(r)) for r in rows]

    # Crear DataFrame seguro
    df = pd.DataFrame(rows[1:], columns=rows[0]) # va generando uno a unu
    dfs.append(df)   # los añade



In [33]:
print("DataFrames creados:", len(dfs))
lista_dataframes = []
for i, df in enumerate(dfs):
    lista_dataframes.append(i)
    print(i, df.shape)
   

print(type(lista_dataframes))
lista_dataframes

DataFrames creados: 2
0 (73, 7)
1 (14, 7)
<class 'list'>


[0, 1]

In [34]:
for x in lista_dataframes:
    print("")
    print(f"----------------------------------dataframe {x}--------------")
    print(dfs[x].head(2))



----------------------------------dataframe 0--------------
                                   Temp.          Sede        Campeón  \
0  Copa de Campeones Europeos de la FIBA                                
1                                   1958  Riga , Sofía  A. S. K. Rīga   

                 Subcampeón     Resultado MVP Final MVP Temporada  
0                                                                  
1  P. B. K. Lukoil Akademik  86–81, 84–71                          

----------------------------------dataframe 1--------------
  Temporada     Sede   Categoría         Jugador             Equipo Dato  \
0   2018–19  Vitoria  Valoración    Shane Larkin  Anadolu Efes S.K.   43   
1   2009–10    París      Puntos  Trajan Langdon      P. B. K. CSKA   32   

              Rival  
0  Fenerbahçe B. S.  
1    K. K. Partizan  


### Solo me interesa el dataframe [0]

In [35]:
print(dfs[0].columns)
dfs[0].head()

Index(['Temp.', 'Sede', 'Campeón', 'Subcampeón', 'Resultado', 'MVP Final',
       'MVP Temporada'],
      dtype='str')


,Temp.,Sede,Campeón,Subcampeón,Resultado,MVP Final,MVP Temporada
0,Copa de Campeones Europeos de la FIBA,,,,,,
1,1958,"Riga , Sofía",A. S. K. Rīga,P. B. K. Lukoil Akademik,"86–81, 84–71",,
2,1958–59,"Riga , Sofía",A. S. K. Rīga,P. B. K. Lukoil Akademik,"79–58, 69–67",,
3,1959–60,"Tiflis , Riga",A. S. K. Rīga,B. C. Dinamo Tbilisi,"86–81, 84–71",,
4,1960–61,"Riga , Moscú",P. B. K. CSKA,A. S. K. Rīga,"87–62, 61–66",,


In [36]:
tabla = tables[0]
filas = tabla.find_all("tr")

import re

anios = []
equipos = []
paises = []

ultimo_anio = None

def extraer_pais(td):
    img = td.find("img")
    if not img:
        return None
    url = img.get("src") or img.get("resource")
    m = re.search(r"Flag_of_([A-Za-z_]+)", url)
    if not m:
        return None

    pais = m.group(1).replace("_", " ")

    # 🔥 quitar "the " al inicio
    pais = re.sub(r"^the\s+", "", pais, flags=re.IGNORECASE)

    return pais

for fila in filas:
    celdas = fila.find_all("td")
    if len(celdas) < 3:
        continue

    # Año está en la columna 0, pero en formato 1957–58
    texto = celdas[0].get_text(strip=True)
    if re.match(r"^\d{4}", texto):   # detecta 1957–58, 1958-59, etc.
        ultimo_anio = int(texto[:4])

    # Campeón está en la columna 2 y tiene bandera
    celda_campeon = celdas[2]
    if not celda_campeon.find("img"):
        continue

    equipos.append(celda_campeon.get_text(strip=True))
    paises.append(extraer_pais(celda_campeon))
    anios.append(ultimo_anio)

print(len(equipos))
print(len(paises))
print(len(anios))



68
68
68


### Saco los equipos de esa tabla

Saco el pais

Creo el dataframe con esas filas

In [37]:
df_total = pd.DataFrame({
    "Año": anios,
    "Equipo": equipos,
    "País": paises
})
df_total.head()


,Año,Equipo,País
0,1958,A. S. K. Rīga,Soviet Union
1,1958,A. S. K. Rīga,Soviet Union
2,1959,A. S. K. Rīga,Soviet Union
3,1960,P. B. K. CSKA,Soviet Union
4,1961,B. C. Dinamo Tbilisi,Soviet Union


Eliminar duplicados

In [38]:
df_sin_duplicados = df_total.drop_duplicates(subset=["Año", "Equipo", "País"])
df_total = df_sin_duplicados
df_total


,Año,Equipo,País
0,1958,A. S. K. Rīga,Soviet Union
2,1959,A. S. K. Rīga,Soviet Union
3,1960,P. B. K. CSKA,Soviet Union
4,1961,B. C. Dinamo Tbilisi,Soviet Union
5,1962,P. B. K. CSKA,Soviet Union
6,1963,Real Madrid Baloncesto,Spain
7,1964,Real Madrid Baloncesto,Spain
8,1965,P. O. Milano,Italy
9,1966,Real Madrid Baloncesto,Spain
10,1967,Real Madrid Baloncesto,Spain


Convertir filas vacias a NaN



In [39]:

df_total = df_total.replace("", np.nan)


### Eliminar filas con algún NA


In [40]:

df_total= df_total.dropna()


### Reindexar el DataFrame completo


In [41]:

df_total = df_total.reset_index(drop=True)
df_total.head()


,Año,Equipo,País
0,1958,A. S. K. Rīga,Soviet Union
1,1959,A. S. K. Rīga,Soviet Union
2,1960,P. B. K. CSKA,Soviet Union
3,1961,B. C. Dinamo Tbilisi,Soviet Union
4,1962,P. B. K. CSKA,Soviet Union



### Función para quitar tildes, acentos y caracteres raros


In [42]:

def limpiar_texto(x):
    if isinstance(x, str):
        x = unicodedata.normalize('NFKD', x)
        x = ''.join(c for c in x if not unicodedata.combining(c))
        x = ' '.join(x.split())
        return x
    return x



# Limpio los caracteres raros


In [43]:
df_campeones = df_total.apply(lambda col: col.map(limpiar_texto))
df_campeones

,Año,Equipo,País
0,1958,A. S. K. Riga,Soviet Union
1,1959,A. S. K. Riga,Soviet Union
2,1960,P. B. K. CSKA,Soviet Union
3,1961,B. C. Dinamo Tbilisi,Soviet Union
4,1962,P. B. K. CSKA,Soviet Union
5,1963,Real Madrid Baloncesto,Spain
6,1964,Real Madrid Baloncesto,Spain
7,1965,P. O. Milano,Italy
8,1966,Real Madrid Baloncesto,Spain
9,1967,Real Madrid Baloncesto,Spain


In [44]:
print(df_campeones.info())
print("-------")
print(df_campeones.nunique())   # no los voy a eliminar , los quiero asi
df_campeones.head()


<class 'pandas.DataFrame'>
RangeIndex: 67 entries, 0 to 66
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Año     67 non-null     int64
 1   Equipo  67 non-null     str  
 2   País    67 non-null     str  
dtypes: int64(1), str(2)
memory usage: 1.7 KB
None
-------
Año       66
Equipo    23
País      11
dtype: int64


,Año,Equipo,País
0,1958,A. S. K. Riga,Soviet Union
1,1959,A. S. K. Riga,Soviet Union
2,1960,P. B. K. CSKA,Soviet Union
3,1961,B. C. Dinamo Tbilisi,Soviet Union
4,1962,P. B. K. CSKA,Soviet Union


codigos_Pais

In [45]:
lista_paises = df_campeones["País"].unique().tolist() # la saco por que me va a valer para la busqueda final
lista_paises

['Soviet Union',
 'Spain',
 'Italy',
 'Israel',
 'Yugoslavia',
 'Serbia and Montenegro',
 'France',
 'Greece',
 'Lithuania',
 'Russia',
 'Turkey']

In [46]:
pd.set_option("display.max_rows", 1000)
pd.set_option("display.max_columns", None)

In [47]:
df_campeones.head()


,Año,Equipo,País
0,1958,A. S. K. Riga,Soviet Union
1,1959,A. S. K. Riga,Soviet Union
2,1960,P. B. K. CSKA,Soviet Union
3,1961,B. C. Dinamo Tbilisi,Soviet Union
4,1962,P. B. K. CSKA,Soviet Union


In [48]:
lista_original = df_campeones.columns.tolist()
lista_nueva = ['year_win','team','statenme']
df_campeones = df_campeones.rename(columns=dict(zip(lista_original, lista_nueva)))
df_campeones.head()

,year_win,team,statenme
0,1958,A. S. K. Riga,Soviet Union
1,1959,A. S. K. Riga,Soviet Union
2,1960,P. B. K. CSKA,Soviet Union
3,1961,B. C. Dinamo Tbilisi,Soviet Union
4,1962,P. B. K. CSKA,Soviet Union


In [49]:
df_campeones.to_csv("Campeones_Copa_Europa_Euroleague.csv", index=False, encoding="utf-8")